# R5: Aggregate Before- vs After-Lesion Comparison

Repeats the report's core cross-session analyses (rewarded vs. unrewarded, the recovery
curve, the template-closeness control) and its per-session trend plots, but for R5 only,
split into two groups instead of chronological periods:

- **Before lesion**: session date < **2026-08-06**
- **After lesion**: session date > **2026-09-08** (i.e. 2026-09-09 onward)

Sessions dated 2026-08-06 through 2026-09-08 (the tail of pre-lesion training plus the
surgery/recovery gap -- see `r5_variability_over_time.ipynb`) are deliberately **excluded**
from both groups, per instruction, for a cleaner two-group comparison.

Sessions with fewer than **10** online-triggered rewards are excluded throughout, matching
the report-wide threshold.

Computed directly per-session via `clirb.native_dannce.build_session_variability` (not
read from the batch CSV), so this reflects whatever sessions are on disk right now.

In [ ]:
import sys, os
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, wilcoxon

from scripts.build_variability_summary import discover_qualifying_sessions, parse_session_date
from clirb.native_dannce import build_session_variability

%matplotlib inline

RAT = 'R5'
MIN_REWARDS = 10
BEFORE_END = '2026-08-06'    # exclusive: date < this
AFTER_START = '2026-09-08'   # exclusive: date > this
GROUP_COLOR = {'before': '#1f77b4', 'after': '#d62728'}

def assign_group(date):
    if date < BEFORE_END:
        return 'before'
    if date > AFTER_START:
        return 'after'
    return None   # excluded transition window

sessions = discover_qualifying_sessions([RAT])
print(f'{len(sessions)} R5 sessions with dual data + behavior log')

rear_rows, session_rows = [], []
for rat, session in sessions:
    date = parse_session_date(session)
    group = assign_group(date)
    if group is None:
        continue
    try:
        d = build_session_variability(rat, session)
    except Exception as e:
        print(f'  {session}: FAILED -- {e!r}')
        continue
    if d['n_rewards'] < MIN_REWARDS:
        continue
    session_rows.append(dict(session=session, session_date=date, group=group,
                             n_rewards=d['n_rewards'], n_valid=d['n_valid']))
    is_rewarded = d['is_rewarded']
    is_tl = d['rear_is_template_like_valid']
    rsr = d['rears_since_reward']
    for i in range(d['n_valid']):
        rear_rows.append(dict(
            session=session, session_date=date, group=group,
            is_rewarded=bool(is_rewarded[i]), is_template_like=bool(is_tl[i]),
            rears_since_reward=int(rsr[i]),
            var_pc1=d['variability'][i, 0], var_pc2=d['variability'][i, 1],
            var_pc3=d['variability'][i, 2],
        ))

df = pd.DataFrame(rear_rows)
sessions_df = pd.DataFrame(session_rows)
for g in ('before', 'after'):
    sub = sessions_df[sessions_df.group == g]
    print(f'{g:8s}: {len(sub)} sessions ({sub.session_date.min()} .. {sub.session_date.max()}), '
         f'{(df.group == g).sum()} rears, {sub.n_rewards.sum()} total rewards')


## Aggregated across sessions

Raw PC units (not session-normalized) -- matching how the single-session before/after
comparison in the report's post-lesion section was done, and reasonable here since R5's
templates share the same underlying PCA basis (pc_weights/feature_means) across
refinements; only the target curve/bounds change.

### 1. Rewarded vs unrewarded rears

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for row_i, g in enumerate(['before', 'after']):
    sub = df[df.group == g]
    for pidx in range(3):
        ax = axes[row_i, pidx]
        col = f'var_pc{pidx + 1}'
        groups = [sub.loc[sub.is_rewarded, col], sub.loc[~sub.is_rewarded, col]]
        bp = ax.boxplot(groups, tick_labels=[f'rewarded\n(n={len(groups[0])})',
                                             f'unrewarded\n(n={len(groups[1])})'],
                        patch_artist=True, showfliers=False)
        for patch, color in zip(bp['boxes'], ['#2ca02c', '#d62728']):
            patch.set_facecolor(color); patch.set_alpha(0.5)
        stat, p = mannwhitneyu(*groups)
        title = f'PC{pidx + 1}  (p={p:.1e})' if row_i == 0 else f'p={p:.1e}'
        ax.set_title(title, fontsize=10)
        if pidx == 0:
            ax.set_ylabel(f'{g}-lesion\nvariability')
fig.suptitle(f'{RAT}: variability by reward presence/absence, before vs after lesion',
            fontsize=12)
fig.tight_layout()
plt.show()


### 1b. Session-level rigor check (correct unit of replication)

In [ ]:
for g in ('before', 'after'):
    sub = df[df.group == g]
    effects = []
    for session, gg in sub.groupby('session'):
        r = gg.loc[gg.is_rewarded, 'var_pc1']
        u = gg.loc[~gg.is_rewarded, 'var_pc1']
        if len(r) and len(u):
            effects.append(u.mean() - r.mean())
    effects = np.array(effects)
    if len(effects) == 0:
        print(f'{g:8s}: no sessions with both rewarded and unrewarded rears')
        continue
    if len(effects) == 1:
        print(f'{g:8s}: n=1 session -- too few for a signed-rank test '
             f'(effect={effects[0]:.2f})')
        continue
    stat, p = wilcoxon(effects)
    print(f'{g:8s}: n={len(effects)} sessions, mean effect '
         f'(unrewarded-rewarded, PC1)={effects.mean():.2f}, '
         f'positive in {int((effects > 0).sum())}/{len(effects)}, Wilcoxon p={p:.2e}')


### 2. Variability vs. rears since last reward (recovery curve)

In [ ]:
BIN_EDGES = [0, 1, 2, 3, 4, 5, np.inf]
BIN_LABELS = ['0', '1', '2', '3', '4', '5+']

fig, axes = plt.subplots(1, 3, figsize=(14, 4.4), sharex=True)
for pidx in range(3):
    ax = axes[pidx]
    col = f'var_pc{pidx + 1}'
    for g in ('before', 'after'):
        sub = df[(df.group == g) & (df.rears_since_reward >= 0)]
        bin_idx = np.digitize(sub['rears_since_reward'], BIN_EDGES[1:-1], right=False)
        means, sems = [], []
        for b in range(len(BIN_LABELS)):
            vals = sub[col][bin_idx == b]
            means.append(vals.mean() if len(vals) else np.nan)
            sems.append(vals.std(ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0.0)
        xs = np.arange(len(BIN_LABELS))
        ax.errorbar(xs, means, yerr=sems, marker='o', ms=4, capsize=3,
                   color=GROUP_COLOR[g], label=f'{g}-lesion')
    ax.set_xticks(xs); ax.set_xticklabels(BIN_LABELS)
    ax.set_xlabel('rears since last reward')
    ax.set_title(f'PC{pidx + 1}')
    if pidx == 0:
        ax.set_ylabel('mean variability +/- SEM')
        ax.legend(fontsize=9)
fig.suptitle(f'{RAT}: variability vs. rears since last reward, before vs after lesion',
            fontsize=12)
fig.tight_layout()
plt.show()


### 3. Template-closeness control (recommendation #2, before vs after)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for row_i, g in enumerate(['before', 'after']):
    sub = df[(df.group == g) & df.is_template_like]
    for pidx in range(3):
        ax = axes[row_i, pidx]
        col = f'var_pc{pidx + 1}'
        groups = [sub.loc[sub.is_rewarded, col], sub.loc[~sub.is_rewarded, col]]
        if min(len(x) for x in groups) < 2:
            ax.set_title('too few rears in one group', fontsize=8)
            continue
        bp = ax.boxplot(groups, tick_labels=[f'rewarded\n(n={len(groups[0])})',
                                             f'unrewarded\n(n={len(groups[1])})'],
                        patch_artist=True, showfliers=False)
        for patch, color in zip(bp['boxes'], ['#2ca02c', '#d62728']):
            patch.set_facecolor(color); patch.set_alpha(0.5)
        stat, p = mannwhitneyu(*groups)
        title = f'PC{pidx + 1}  (p={p:.1e})' if row_i == 0 else f'p={p:.1e}'
        ax.set_title(title, fontsize=10)
        if pidx == 0:
            ax.set_ylabel(f'{g}-lesion\nvariability')
fig.suptitle(f'{RAT}: template-like rears only, variability by reward, before vs after lesion',
            fontsize=12)
fig.tight_layout()
plt.show()


## Metric changing across sessions

Per-session trend, shown as two separate mini-timelines (before / after) rather than one
continuous axis -- the ~5-week gap between the groups would otherwise squash both into
illegible slivers. Top: mean PC1 variability per session (all rears). Bottom: the reward
effect per session (mean unrewarded - mean rewarded, PC1) -- the same metric as the
report's Fig. 9 bottom row.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7.5), sharex=False)
for col_i, g in enumerate(['before', 'after']):
    sub_sessions = sessions_df[sessions_df.group == g].sort_values('session_date')
    rows = []
    for session in sub_sessions.session:
        gg = df[df.session == session]
        r = gg.loc[gg.is_rewarded, 'var_pc1']
        u = gg.loc[~gg.is_rewarded, 'var_pc1']
        rows.append(dict(session=session, session_date=gg.session_date.iloc[0],
                         mean_var_pc1=gg['var_pc1'].mean(),
                         reward_effect=(u.mean() - r.mean()) if len(r) and len(u) else np.nan))
    trend = pd.DataFrame(rows).sort_values('session_date').reset_index(drop=True)
    xs = np.arange(len(trend))

    axes[0, col_i].plot(xs, trend.mean_var_pc1, marker='o', ms=4, color=GROUP_COLOR[g])
    axes[0, col_i].set_title(f'{g}-lesion  (n={len(trend)} sessions)', color=GROUP_COLOR[g])
    axes[0, col_i].set_xticks(xs)
    axes[0, col_i].set_xticklabels(trend.session_date, rotation=45, ha='right', fontsize=7)

    axes[1, col_i].plot(xs, trend.reward_effect, marker='o', ms=4, color=GROUP_COLOR[g])
    axes[1, col_i].axhline(0.0, color='k', lw=0.8, ls=':')
    axes[1, col_i].set_xticks(xs)
    axes[1, col_i].set_xticklabels(trend.session_date, rotation=45, ha='right', fontsize=7)
    axes[1, col_i].set_xlabel('session date')

axes[0, 0].set_ylabel('mean PC1 variability\n(raw)')
axes[1, 0].set_ylabel('reward effect on PC1\n(unrewarded - rewarded)')
fig.suptitle(f'{RAT}: per-session trend, before vs after lesion', fontsize=12)
fig.tight_layout()
plt.show()
